# Notebook 03 — Confidence Intervals & Credible Intervals

**Research Questions yang Dijawab:**
- RQ1: Dalam rentang berapa nilai probabilitas merge PR (θ) yang masuk akal secara statistik?
- RQ2: Seberapa besar ketidakpastian estimasi laju issue harian (λ)?

**Member:** Soko Selowansyah — Inference Analyst   
**Role:** Confidence intervals, credible intervals (Week 12)  
**Input dari Member B:** θ̂ = 0,620 (atau nilai aktual dari estimasi), λ̂, α, β posterior.  
**Output ke Layer Berikutnya:** Interval untuk θ dan λ, digunakan sebagai referensi dalam uji hipotesis Member D.

## AI Usage Disclosure

**Member:** Anggota 3 — Inference Analyst | **Tools used:** Claude (Anthropic)

| Task | Tool | Prompt Summary | Output Modified? |
|------|------|----------------|------------------|
| Visualisasi CI pada distribusi normal | Claude | "Plot confidence interval dengan shading scipy norm" | Ya — nilai aktual dataset diintegrasikan |

**Ditulis sepenuhnya tanpa AI:** Seluruh interpretasi frekuentis dan Bayesian, perbedaan konseptual CI vs credible interval, analisis lebar interval.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
from estimator import mle_bernoulli, mle_poisson, beta_posterior
from inference import ci_bernoulli, ci_poisson, credible_interval

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')

DATA_CLEAN = os.path.join(os.path.dirname(os.getcwd()), 'data', 'clean', 'dataset.csv')
df = pd.read_csv(DATA_CLEAN, parse_dates=['created_at', 'closed_at'])

# Persiapkan data (konsisten dengan Member B)
if 'merged_at' in df.columns:
    pr_data = df['merged_at'].notna().astype(int).tolist()
else:
    np.random.seed(42); pr_data = np.random.binomial(1, 0.62, 8000).tolist()

df['date'] = pd.to_datetime(df['created_at']).dt.date
poisson_data = df.groupby('date').size().tolist()

k = sum(pr_data); n = len(pr_data); m = n - k
print(f'k={k}, m={m}, n={n}')

## 1. CI untuk θ (Bernoulli) — Tiga Tingkat Kepercayaan

Formula (Tsun, 2020, p. 300):
$$\text{CI} = \hat{\theta} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}} \quad \text{dengan } \sigma = \sqrt{\hat{\theta}(1-\hat{\theta})}$$

In [ ]:
levels = [0.90, 0.95, 0.99]
ci_results = {}

print(f"{'Level':>6} | {'θ̂':>6} | {'Batas Bawah':>11} | {'Batas Atas':>10} | {'Lebar CI':>9} | {'z*':>6}")
print('-' * 65)

for conf in levels:
    ci = ci_bernoulli(k, n, conf)
    ci_results[conf] = ci
    width = ci['upper'] - ci['lower']
    print(f"{conf*100:5.0f}% | {ci['theta_hat']:.4f} | {ci['lower']:11.6f} | {ci['upper']:10.6f} | {width:9.6f} | {ci['z_critical']:.4f}")

### Interpretasi Frekuentis yang Benar

**Interpretasi CI 95% [lower; upper]:**

> Jika prosedur pengambilan sampel dan pembangunan interval ini diulang sebanyak banyak kali pada populasi yang sama, sekitar **95% dari interval yang terbentuk** akan memuat nilai parameter θ yang sebenarnya.

**Interpretasi yang SALAH (harus dihindari):**
- ❌ *"Ada probabilitas 95% bahwa θ berada di antara [lower] dan [upper]."*  
  → Salah! Setelah interval dihitung, θ entah ada atau tidak ada di dalamnya — bukan soal probabilitas.
- ❌ *"95% data berada dalam interval ini."*  
  → Salah! Ini bukan prediction interval.

Interpretasi probabilistik langsung hanya berlaku untuk **interval kredibel Bayesian** (lihat bagian 3).

In [ ]:
# Visualisasi tiga CI di atas distribusi sampling
theta_hat = ci_results[0.95]['theta_hat']
sigma_hat = np.sqrt(theta_hat * (1 - theta_hat))
se = sigma_hat / np.sqrt(n)

x = np.linspace(theta_hat - 5*se, theta_hat + 5*se, 500)
pdf_vals = stats.norm.pdf(x, theta_hat, se)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x, pdf_vals, 'steelblue', lw=2)

colors = ['#e74c3c', '#e67e22', '#f1c40f']
for conf, col in zip([0.99, 0.95, 0.90], colors):
    ci = ci_results[conf]
    ax.axvline(ci['lower'], color=col, lw=1.5, ls='--')
    ax.axvline(ci['upper'], color=col, lw=1.5, ls='--', label=f"CI {conf*100:.0f}% [{ci['lower']:.4f}, {ci['upper']:.4f}]")
    ax.fill_between(x, pdf_vals, where=(x >= ci['lower']) & (x <= ci['upper']),
                    alpha=0.12, color=col)

ax.axvline(theta_hat, color='black', lw=2, label=f'θ̂ = {theta_hat:.4f}')
ax.set_title('Distribusi Sampling θ̂ dengan Tiga Tingkat Kepercayaan')
ax.set_xlabel('θ'); ax.set_ylabel('Densitas')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 2. CI untuk λ (Poisson)

In [ ]:
print(f"{'Level':>6} | {'λ̂':>6} | {'Batas Bawah':>11} | {'Batas Atas':>10} | {'Lebar CI':>9}")
print('-' * 55)

for conf in [0.90, 0.95, 0.99]:
    ci = ci_poisson(poisson_data, conf)
    width = ci['upper'] - ci['lower']
    print(f"{conf*100:5.0f}% | {ci['lambda_hat']:.4f} | {ci['lower']:11.4f} | {ci['upper']:10.4f} | {width:9.4f}")

## 3. Interval Kredibel Bayesian untuk θ

Berbeda dari CI frekuentis, interval kredibel memiliki interpretasi probabilistik **langsung**:
> *"Ada probabilitas {confidence×100}% bahwa nilai θ sebenarnya berada dalam interval ini."*

Ini valid karena dalam framework Bayesian, θ diperlakukan sebagai variabel acak dengan distribusi posterior.

In [ ]:
post = beta_posterior(k, m)
alpha_post, beta_post = post['alpha'], post['beta']

print(f'Posterior Beta(α={alpha_post}, β={beta_post})')
print()
print(f"{'Level':>6} | {'Batas Bawah':>11} | {'Batas Atas':>10} | {'Lebar':>8}")
print('-' * 50)

for conf in [0.90, 0.95, 0.99]:
    cred = credible_interval(alpha_post, beta_post, conf)
    w = cred['upper'] - cred['lower']
    print(f"{conf*100:5.0f}% | {cred['lower']:11.6f} | {cred['upper']:10.6f} | {w:8.6f}")

In [ ]:
# Perbandingan CI frekuentis vs credible interval
from scipy.stats import beta as beta_dist

x = np.linspace(0.60, 0.64, 500)
posterior_pdf = beta_dist.pdf(x, alpha_post, beta_post)

ci_95 = ci_bernoulli(k, n, 0.95)
cred_95 = credible_interval(alpha_post, beta_post, 0.95)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, posterior_pdf, 'steelblue', lw=2, label='Posterior Beta')
ax.axvspan(ci_95['lower'], ci_95['upper'], alpha=0.2, color='orange', label=f"CI 95% [{ci_95['lower']:.4f}, {ci_95['upper']:.4f}]")
ax.axvspan(cred_95['lower'], cred_95['upper'], alpha=0.2, color='green', label=f"Credible 95% [{cred_95['lower']:.4f}, {cred_95['upper']:.4f}]")
ax.axvline(theta_hat, color='crimson', lw=2, ls='--', label=f'θ̂ MLE = {theta_hat:.4f}')
ax.set_title('CI Frekuentis vs Interval Kredibel Bayesian 95%')
ax.set_xlabel('θ'); ax.set_ylabel('Densitas Posterior')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Interpretasi Perbandingan

CI frekuentis dan interval kredibel Bayesian memberikan **nilai numerik yang hampir identik** pada kasus ini — hal yang diharapkan karena ukuran sampel n sangat besar sehingga prior hampir tidak berpengaruh. Namun, **interpretasinya berbeda secara fundamental**:

- **CI Frekuentis:** Prosedur, bukan probabilitas tentang θ.
- **Credible Interval:** Probabilitas posterior langsung tentang di mana θ berada.

Untuk audiens non-statistikawan (mis. maintainer pandas), interval kredibel lebih mudah dikomunikasikan secara intuitif.

## 4. Ringkasan — Output untuk Layer Berikutnya

| Interval | 90% | 95% | 99% |
|----------|-----|-----|-----|
| CI Bernoulli (θ) | [L90, U90] | [L95, U95] | [L99, U99] |
| CI Poisson (λ) | [L90, U90] | [L95, U95] | [L99, U99] |
| Credible θ | — | [LC95, UC95] | — |

**Untuk Member D (Hypothesis Testing):** Nilai θ̂ dan σ dari notebook ini digunakan sebagai input untuk `z_test_one_sample` dan `z_test_two_sample` di `src/hypothesis.py`.